# 🧪 Simple Deployment Validator

**Streamlined validation that mirrors the deployer notebooks**

This notebook validates:
1. **Notebooks** - From dist path → Workspace (with prefixes)
2. **Lakehouses** - From manifest → Workspace (with prefixes)
3. **Tables** - From manifest → Lakehouse tables
4. **Pipelines** - From dist path → Workspace (with prefixes, no placeholders)

---

## Setup

In [ ]:
%run common_deployment_config

In [ ]:
import json
import re
from collections import defaultdict

print("✓ Simple Deployment Validator Initialized")
print(f"\n📊 Configuration:")
print(f"   Workspace ID: {WORKSPACE_ID}")
print(f"   Artifact Version: {ARTIFACT_VERSION}")
print(f"   Company Prefix: '{COMPANY_PREFIX}'")
print(f"   Technical Prefix: '{TECHNICAL_PREFIX}'")
print(f"   Base Dist Path: {BASE_DIST_PATH}")

---
## 1️⃣ Notebook Validation

**Mirrors notebook_deployer logic:**
- Scans: `{BASE_DIST_PATH}/healthcare-artifacts/{ARTIFACT_VERSION}/{capability}/Notebooks/*.ipynb`
- Applies prefix: `msft_config.ipynb` → `healthcare_msft_config`
- Checks if exists in workspace

In [ ]:
print("="*80)
print("1️⃣ NOTEBOOK VALIDATION")
print("="*80)

# Step 1: Scan dist folder (same as notebook_deployer)
healthcare_path = f"{BASE_DIST_PATH}/healthcare-artifacts/{ARTIFACT_VERSION}"
print(f"\n📂 Scanning: {healthcare_path}")

source_notebooks = []

try:
    capabilities = mssparkutils.fs.ls(healthcare_path)
    
    for cap_item in capabilities:
        if not cap_item.isDir:
            continue
        
        capability = cap_item.name
        notebooks_folder = f"{cap_item.path}/Notebooks"
        
        if not mssparkutils.fs.exists(notebooks_folder):
            continue
        
        try:
            notebook_files = [
                f for f in mssparkutils.fs.ls(notebooks_folder)
                if f.name.endswith(".ipynb")
            ]
            
            for nb_file in notebook_files:
                # Apply prefix (same logic as notebook_deployer)
                expected_name = build_notebook_display_name(nb_file.name)
                
                source_notebooks.append({
                    'source_file': nb_file.name,
                    'capability': capability,
                    'expected_deployed_name': expected_name
                })
        except Exception as e:
            print(f"  Warning: Error scanning {capability}: {e}")
            
    print(f"\n✓ Found {len(source_notebooks)} notebooks in dist folder")
    
    # Group by capability
    by_cap = defaultdict(int)
    for nb in source_notebooks:
        by_cap[nb['capability']] += 1
    
    print(f"\n📋 By capability:")
    for cap in sorted(by_cap.keys()):
        print(f"   • {cap}: {by_cap[cap]} notebook(s)")
        
except Exception as e:
    print(f"\n✗ ERROR scanning dist folder: {e}")
    source_notebooks = []

In [ ]:
# Step 2: Get deployed notebooks from workspace
print(f"\n📦 Getting deployed notebooks from workspace...")

try:
    client = FabricRestClient()
    response = client.get(f"/v1/workspaces/{WORKSPACE_ID}/notebooks")
    
    if response.status_code == 200:
        notebooks = response.json().get('value', [])
        deployed_names = set(nb['displayName'].lower() for nb in notebooks)
        deployed_map = {nb['displayName'].lower(): nb['displayName'] for nb in notebooks}
        
        print(f"✓ Found {len(deployed_names)} notebooks in workspace")
    else:
        print(f"\n✗ ERROR: HTTP {response.status_code}")
        deployed_names = set()
        deployed_map = {}
    
except Exception as e:
    print(f"\n✗ ERROR getting deployed notebooks: {e}")
    deployed_names = set()
    deployed_map = {}

In [ ]:
# Step 3: Compare source → workspace
print(f"\n🔍 Validation: Source → Workspace\n")
print("="*80)

found = []
missing = []

for nb in source_notebooks:
    expected_lower = nb['expected_deployed_name'].lower()
    
    if expected_lower in deployed_names:
        actual_name = deployed_map[expected_lower]
        found.append(nb)
        print(f"✅ {nb['capability']}")
        print(f"   Source:    {nb['source_file']}")
        print(f"   Workspace: {actual_name}")
        print(f"   Mapping:   {nb['source_file']} → {actual_name}")
        print()
    else:
        missing.append(nb)
        print(f"❌ {nb['capability']}")
        print(f"   Source:   {nb['source_file']}")
        print(f"   Expected: {nb['expected_deployed_name']}")
        print(f"   Status:   NOT FOUND IN WORKSPACE")
        print()

print("="*80)
print(f"\n📊 Summary:")
print(f"   Total notebooks in dist: {len(source_notebooks)}")
print(f"   ✅ Found in workspace: {len(found)}")
print(f"   ❌ Missing: {len(missing)}")

if missing:
    print(f"\n⚠️  Missing notebooks by capability:")
    by_cap = defaultdict(list)
    for nb in missing:
        by_cap[nb['capability']].append(nb['source_file'])
    
    for cap in sorted(by_cap.keys()):
        print(f"\n   {cap}:")
        for name in sorted(by_cap[cap]):
            print(f"      • {name}")

---
## 2️⃣ Lakehouse Validation

**From LakehouseHydrationManifest.json:**
- Reads lakehouse keys: `bronze`, `silver`, `omop`, etc.
- Applies prefix: `bronze` → `healthcare_msft_bronze`
- Checks if exists in workspace

In [ ]:
print("="*80)
print("2️⃣ LAKEHOUSE VALIDATION")
print("="*80)

# Step 1: Load manifest
manifest_path = f"{BASE_DIST_PATH}/healthcare-configuration/{ARTIFACT_VERSION}/system-configurations/LakehouseHydrationManifest.json"
print(f"\n📄 Loading: {manifest_path}")

try:
    manifest_content = mssparkutils.fs.head(manifest_path, 1000000)
    lakehouse_manifest = json.loads(manifest_content)
    
    print(f"\n✓ Loaded {len(lakehouse_manifest)} lakehouses from manifest")
    print(f"\n📋 Lakehouses in manifest:")
    for lh_key, lh_config in lakehouse_manifest.items():
        table_count = len(lh_config.get('tables', []))
        print(f"   • {lh_key}: {table_count} table(s)")
        
except Exception as e:
    print(f"\n✗ ERROR loading manifest: {e}")
    lakehouse_manifest = {}

In [ ]:
# Step 2: Get deployed lakehouses
print(f"\n📦 Getting deployed lakehouses from workspace...")

try:
    client = FabricRestClient()
    response = client.get(f"/v1/workspaces/{WORKSPACE_ID}/lakehouses")
    
    if response.status_code == 200:
        lakehouses = response.json().get('value', [])
        deployed_lh_names = set(lh['displayName'].lower() for lh in lakehouses)
        deployed_lh_map = {lh['displayName'].lower(): lh['displayName'] for lh in lakehouses}
        
        print(f"✓ Found {len(deployed_lh_names)} lakehouses in workspace")
    else:
        print(f"\n✗ ERROR: HTTP {response.status_code}")
        deployed_lh_names = set()
        deployed_lh_map = {}
    
except Exception as e:
    print(f"\n✗ ERROR getting deployed lakehouses: {e}")
    deployed_lh_names = set()
    deployed_lh_map = {}

In [ ]:
# Step 3: Compare manifest → workspace
print(f"\n🔍 Validation: Manifest → Workspace\n")
print("="*80)

found_lakehouses = []
missing_lakehouses = []

for lh_key, lh_config in lakehouse_manifest.items():
    # Replace hyphens with underscores, then apply prefix (same as deployer)
    lh_key_normalized = lh_key.replace('-', '_')
    expected_name = build_artifact_name(lh_key_normalized)
    expected_lower = expected_name.lower()
    table_count = len(lh_config.get('tables', []))
    
    if expected_lower in deployed_lh_names:
        actual_name = deployed_lh_map[expected_lower]
        found_lakehouses.append({
            'manifest_key': lh_key,
            'deployed_name': actual_name,
            'table_count': table_count,
            'tables': lh_config.get('tables', [])
        })
        print(f"✅ Manifest: {lh_key}")
        print(f"   Workspace: {actual_name}")
        print(f"   Mapping:   {lh_key} → {actual_name}")
        print(f"   Tables:    {table_count}")
        print()
    else:
        missing_lakehouses.append(lh_key)
        print(f"❌ Manifest: {lh_key}")
        print(f"   Expected:  {expected_name}")
        print(f"   Status:    NOT FOUND IN WORKSPACE")
        print()

print("="*80)
print(f"\n📊 Summary:")
print(f"   Total lakehouses in manifest: {len(lakehouse_manifest)}")
print(f"   ✅ Found in workspace: {len(found_lakehouses)}")
print(f"   ❌ Missing: {len(missing_lakehouses)}")

if missing_lakehouses:
    print(f"\n⚠️  Missing lakehouses:")
    for lh_key in missing_lakehouses:
        print(f"      • {lh_key}")

---
## 3️⃣ Table Validation

**For each lakehouse (with prefix):**
- Gets expected tables from manifest
- Queries lakehouse: `SHOW TABLES IN {lakehouse_name}`
- Compares expected vs actual

In [ ]:
print("="*80)
print("3️⃣ TABLE VALIDATION")
print("="*80)

if not found_lakehouses:
    print("\n⚠️  No lakehouses found - skipping table validation")
else:
    print(f"\n🔍 Validating tables in {len(found_lakehouses)} lakehouse(es)\n")
    print("="*80)
    
    total_expected_tables = 0
    total_found_tables = 0
    total_missing_tables = 0
    
    for lh in found_lakehouses:
        lakehouse_name = lh['deployed_name']
        expected_tables = lh['tables']
        
        if not expected_tables:
            print(f"\nℹ️  {lh['manifest_key']} ({lakehouse_name})")
            print(f"   No tables defined in manifest - skipping")
            print()
            continue
        
        print(f"\n📦 {lh['manifest_key']} → {lakehouse_name}")
        print(f"   Expected tables: {len(expected_tables)}")
        
        try:
            # Query tables from lakehouse
            tables_df = spark.sql(f"SHOW TABLES IN {lakehouse_name}")
            deployed_tables = [row.tableName for row in tables_df.collect()]
            deployed_tables_lower = set(t.lower() for t in deployed_tables)
            
            print(f"   Deployed tables: {len(deployed_tables)}")
            
            # Compare
            expected_tables_lower = set(t.lower() for t in expected_tables)
            missing_tables = []
            found_tables = []
            
            for table_name in expected_tables:
                if table_name.lower() in deployed_tables_lower:
                    found_tables.append(table_name)
                else:
                    missing_tables.append(table_name)
            
            # Update counts
            total_expected_tables += len(expected_tables)
            total_found_tables += len(found_tables)
            total_missing_tables += len(missing_tables)
            
            # Report
            if missing_tables:
                print(f"   ❌ Missing: {len(missing_tables)} table(s)")
                for table in missing_tables[:5]:
                    print(f"      • {table}")
                if len(missing_tables) > 5:
                    print(f"      ... and {len(missing_tables) - 5} more")
            else:
                print(f"   ✅ All expected tables found")
                
        except Exception as e:
            print(f"   ✗ Error reading tables: {e}")
            total_expected_tables += len(expected_tables)
    
    print("\n" + "="*80)
    print(f"\n📊 Overall Summary:")
    print(f"   Total expected tables: {total_expected_tables}")
    print(f"   ✅ Found: {total_found_tables}")
    print(f"   ❌ Missing: {total_missing_tables}")

---
## 4️⃣ Pipeline Validation

**Mirrors pipeline_deployer logic:**
- Scans: `{BASE_DIST_PATH}/healthcare-artifacts/{ARTIFACT_VERSION}/{capability}/DataPipelines/*.json`
- Applies prefix: `msft_claims_ingestion.json` → `healthcare_msft_claims_ingestion`
- Checks if exists in workspace
- Validates no placeholders remain

In [ ]:
print("="*80)
print("4️⃣ PIPELINE VALIDATION")
print("="*80)

# Step 1: Scan dist folder (same as pipeline_deployer)
healthcare_path = f"{BASE_DIST_PATH}/healthcare-artifacts/{ARTIFACT_VERSION}"
print(f"\n📂 Scanning: {healthcare_path}")

source_pipelines = []

try:
    capabilities = mssparkutils.fs.ls(healthcare_path)
    
    for cap_item in capabilities:
        if not cap_item.isDir:
            continue
        
        capability = cap_item.name
        pipelines_folder = f"{cap_item.path}/DataPipelines"
        
        if not mssparkutils.fs.exists(pipelines_folder):
            continue
        
        try:
            pipeline_files = [
                f for f in mssparkutils.fs.ls(pipelines_folder)
                if f.name.endswith(".json")
            ]
            
            for pipeline_file in pipeline_files:
                # Apply prefix (same logic as pipeline_deployer)
                base_name = pipeline_file.name.replace('.json', '')
                
                # Smart prefix handling: if name already has technical prefix, add only company prefix
                expected_name = build_artifact_name(base_name)
                
                tech_prefix_str = TECHNICAL_PREFIX.strip() if TECHNICAL_PREFIX else ""
                if tech_prefix_str and base_name.startswith(f"{tech_prefix_str}_"):
                    # Has tech prefix, add only company prefix
                    parts = []
                    if COMPANY_PREFIX and COMPANY_PREFIX.strip():
                        parts.append(COMPANY_PREFIX.strip())
                    parts.append(base_name)
                    expected_name = "_".join(parts) if parts else base_name
                
                source_pipelines.append({
                    'source_file': pipeline_file.name,
                    'source_path': pipeline_file.path,
                    'capability': capability,
                    'expected_deployed_name': expected_name
                })
        except Exception as e:
            print(f"  Warning: Error scanning {capability}: {e}")
            
    print(f"\n✓ Found {len(source_pipelines)} pipelines in dist folder")
    
    # Group by capability
    by_cap = defaultdict(int)
    for pipeline in source_pipelines:
        by_cap[pipeline['capability']] += 1
    
    print(f"\n📋 By capability:")
    for cap in sorted(by_cap.keys()):
        print(f"   • {cap}: {by_cap[cap]} pipeline(s)")
        
except Exception as e:
    print(f"\n✗ ERROR scanning dist folder: {e}")
    source_pipelines = []

In [ ]:
# Step 2: Get deployed pipelines from workspace
print(f"\n📦 Getting deployed pipelines from workspace...")

try:
    client = FabricRestClient()
    response = client.get(f"/v1/workspaces/{WORKSPACE_ID}/items?type=DataPipeline")
    
    if response.status_code == 200:
        pipelines = response.json().get('value', [])
        deployed_pipeline_names = set(p['displayName'].lower() for p in pipelines)
        deployed_pipeline_map = {p['displayName'].lower(): p['displayName'] for p in pipelines}
        
        print(f"✓ Found {len(deployed_pipeline_names)} pipelines in workspace")
    else:
        print(f"\n✗ ERROR: HTTP {response.status_code}")
        deployed_pipeline_names = set()
        deployed_pipeline_map = {}
        
except Exception as e:
    print(f"\n✗ ERROR getting deployed pipelines: {e}")
    deployed_pipeline_names = set()
    deployed_pipeline_map = {}

In [ ]:
# Step 3: Compare source → workspace
print(f"\n🔍 Validation: Source → Workspace\n")
print("="*80)

found_pipelines = []
missing_pipelines = []

for pipeline in source_pipelines:
    expected_lower = pipeline['expected_deployed_name'].lower()
    
    if expected_lower in deployed_pipeline_names:
        actual_name = deployed_pipeline_map[expected_lower]
        found_pipelines.append(pipeline)
        print(f"✅ {pipeline['capability']}")
        print(f"   Source:    {pipeline['source_file']}")
        print(f"   Workspace: {actual_name}")
        print(f"   Mapping:   {pipeline['source_file']} → {actual_name}")
        print()
    else:
        missing_pipelines.append(pipeline)
        print(f"❌ {pipeline['capability']}")
        print(f"   Source:   {pipeline['source_file']}")
        print(f"   Expected: {pipeline['expected_deployed_name']}")
        print(f"   Status:   NOT FOUND IN WORKSPACE")
        print()

print("="*80)
print(f"\n📊 Summary:")
print(f"   Total pipelines in dist: {len(source_pipelines)}")
print(f"   ✅ Found in workspace: {len(found_pipelines)}")
print(f"   ❌ Missing: {len(missing_pipelines)}")

if missing_pipelines:
    print(f"\n⚠️  Missing pipelines by capability:")
    by_cap = defaultdict(list)
    for pipeline in missing_pipelines:
        by_cap[pipeline['capability']].append(pipeline['source_file'])
    
    for cap in sorted(by_cap.keys()):
        print(f"\n   {cap}:")
        for name in sorted(by_cap[cap]):
            print(f"      • {name}")

In [ ]:
# Step 4: Check for placeholders in deployed pipelines
print(f"\n🔍 Checking for placeholders in DEPLOYED pipelines (not source)...\n")
print("="*80)

pipelines_with_placeholders = []
placeholder_pattern = re.compile(r'%%[^%]+%%')

for pipeline in found_pipelines:
    try:
        # Get the deployed pipeline definition from workspace
        pipeline_name = pipeline['expected_deployed_name']
        
        # Get pipeline ID first
        client = FabricRestClient()
        response = client.get(f"/v1/workspaces/{WORKSPACE_ID}/items?type=DataPipeline")
        
        if response.status_code == 200:
            pipelines_list = response.json().get('value', [])
            pipeline_obj = next((p for p in pipelines_list if p['displayName'].lower() == pipeline_name.lower()), None)
            
            if pipeline_obj:
                pipeline_id = pipeline_obj['id']
                
                # Get pipeline definition
                def_response = client.get(f"/v1/workspaces/{WORKSPACE_ID}/dataPipelines/{pipeline_id}")
                
                if def_response.status_code == 200:
                    # Convert to string to search for placeholders
                    content = str(def_response.json())
                    
                    # Search for placeholder patterns
                    placeholders_found = placeholder_pattern.findall(content)
                    
                    if placeholders_found:
                        unique_placeholders = list(set(placeholders_found))
                        pipelines_with_placeholders.append({
                            'pipeline': pipeline['source_file'],
                            'deployed_name': pipeline_name,
                            'capability': pipeline['capability'],
                            'placeholders': unique_placeholders
                        })
                        print(f"⚠️  {pipeline['capability']} / {pipeline_name}")
                        print(f"   Placeholders in DEPLOYED pipeline:")
                        for ph in unique_placeholders:
                            print(f"      • {ph}")
                        print()
                else:
                    print(f"⚠️  Could not get definition for {pipeline_name} (HTTP {def_response.status_code})")
            else:
                print(f"⚠️  Could not find pipeline object for {pipeline_name}")
        else:
            print(f"⚠️  Error listing pipelines (HTTP {response.status_code})")
            
    except Exception as e:
        print(f"⚠️  Error checking {pipeline['source_file']}: {e}")
        print()

print("="*80)
print(f"\n📊 Placeholder Check Summary:")
if pipelines_with_placeholders:
    print(f"   ⚠️  {len(pipelines_with_placeholders)} DEPLOYED pipeline(s) have unreplaced placeholders")
    print(f"   (These are in the workspace, not in source files)")
else:
    print(f"   ✅ No placeholders found in any deployed pipeline")

In [ ]:
# -----------------------------------------------------------------------------
# 4️⃣ Power BI Artifacts Validation (Semantic Models + Reports)
# This uses the canonical lists provided by common_deployment_config (SEMANTIC_MODELS, REPORTS)
# -----------------------------------------------------------------------------

import re
from collections import Counter

# Initialize Fabric client
try:
    client = FabricRestClient()
except Exception as e:
    raise RuntimeError(f"Cannot create FabricRestClient: {e}")

def _find_item(display_name: str, item_type: str):
    resp = client.get(f"/v1/workspaces/{WORKSPACE_ID}/items?type={item_type}")
    resp.raise_for_status()
    for it in resp.json().get("value", []):
        if it.get("displayName") == display_name:
            return it
    return None

def _check_name(name: str, item_type: str):
    prefixed = build_artifact_name(name)
    pref_item = _find_item(prefixed, item_type)
    raw_item = _find_item(name, item_type)
    if pref_item:
        return ("prefixed", pref_item.get("id"), pref_item.get("displayName"))
    if raw_item:
        return ("raw", raw_item.get("id"), raw_item.get("displayName"))
    return ("missing", None, None)

def validate_powerbi_artifacts():
    print("=" * 80)
    print("4️⃣ POWER BI ARTIFACTS VALIDATION")
    print("=" * 80)

    # Derive expected names from shared canonical config (common_deployment_config)
    expected_semantic_models = [m['name'] if isinstance(m, dict) else m for m in SEMANTIC_MODELS]
    expected_reports = [r['name'] if isinstance(r, dict) else r for r in REPORTS]

    results = []
    for sm in expected_semantic_models:
        status, iid, disp = _check_name(sm, "SemanticModel")
        results.append({"Name": sm, "Type": "SemanticModel", "Status": status, "ID": iid, "DisplayName": disp})
        print(f"{sm}: {status} {disp or ''}")

    for rpt in expected_reports:
        status, iid, disp = _check_name(rpt, "Report")
        results.append({"Name": rpt, "Type": "Report", "Status": status, "ID": iid, "DisplayName": disp})
        print(f"{rpt}: {status} {disp or ''}")

    print("=" * 80)
    print("SUMMARY:")
    cnt = Counter(r["Status"] for r in results)
    print(f"  prefixed: {cnt.get('prefixed', 0)}")
    print(f"  raw:      {cnt.get('raw', 0)}")
    print(f"  missing:  {cnt.get('missing', 0)}")

    return results
print(validate_powerbi_artifacts())

---
## ✅ Validation Complete

All validations finished. Review the results above:

1. ✅ **Notebooks** - Source files → Workspace (with prefixes)
2. ✅ **Lakehouses** - Manifest keys → Workspace (with prefixes)
3. ✅ **Tables** - Manifest definitions → Lakehouse tables
4. ✅ **Pipelines** - Source files → Workspace (with prefixes, placeholder check)

---